# 03 - Ejercicios Prácticos: Gradient Boosting Classifiers (XGBoost, LightGBM, CatBoost)

## Curso de Machine Learning: Modelos de Clasificación

En este cuaderno pondrás en práctica el cálculo de pseudo-residuos en clasificación, la configuración de modelos secuenciales de **Gradient Boosting**, la parada temprana (*Early Stopping*), el manejo de clases desbalanceadas y la sintonización de hiperparámetros a través de 5 ejercicios aplicados.

---

### Instrucciones
1. Ejecuta la celda de configuración inicial.
2. Completa el código en las secciones señaladas con `# TU CODIGO AQUI`.
3. Verifica tus resultados con las métricas y gráficas solicitadas.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine, load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score, f1_score, classification_report, ConfusionMatrixDisplay

np.random.seed(42)
print("Entorno de ejercicios Gradient Boosting preparado.")


---
### Ejercicio 1: Cálculo Manual de Log-Odds, Probabilidades Sigmoides y Pseudo-Residuos

Dado un conjunto pequeño de 5 muestras con etiquetas reales `y = np.array([1, 0, 1, 1, 0])`:
1. Calcula la proporción de la clase positiva $\bar{y}$ y el **log-odds inicial** $F_0 = \ln\left(\frac{\bar{y}}{1 - \bar{y}}\right)$.
2. Convierte $F_0$ de vuelta a probabilidad usando la función sigmoide $\hat{p}_0 = \frac{1}{1 + e^{-F_0}}$.
3. Calcula el vector de **pseudo-residuos iniciales** $r_{i,1} = y_i - \hat{p}_0$ sobre el cual se entrenará el primer árbol del ensamble.
4. Calcula el **Hessiano inicial** para cada muestra: $H_{i,1} = \hat{p}_0 (1 - \hat{p}_0)$.


In [ ]:
y_ejemplo = np.array([1, 0, 1, 1, 0])

# TU CODIGO AQUI
# 1. Calcular proporcion media y log-odds F_0
# 2. Aplicar funcion sigmoide para obtener p_0
# 3. Calcular pseudo-residuos (y_ejemplo - p_0)
# 4. Calcular Hessiano H_0 = p_0 * (1 - p_0) e imprimir resultados


---
### Ejercicio 2: Clasificación Multiclase en Dataset Wine y Evolución de `staged_predict_proba`

1. Carga `load_wine()` y divide los datos en entrenamiento y prueba ($75\% / 25\%$, estratificado con `random_state=42`).
2. Entrena un `GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)`.
3. Utilizando `staged_predict_proba(X_test)`, calcula el `log_loss` multiclase en cada una de las 100 iteraciones.
4. Grafica la curva de `log_loss` en prueba e imprime el número de árbol que minimiza la pérdida.


In [ ]:
# TU CODIGO AQUI
# 1. Cargar load_wine y hacer train_test_split estratificado
# 2. Entrenar GradientBoostingClassifier
# 3. Evaluar log_loss con staged_predict_proba en X_test
# 4. Graficar la evolucion del Log-Loss e imprimir el optimo


---
### Ejercicio 3: Parada Temprana (*Early Stopping*) y Regularización $L_2$ con `HistGradientBoostingClassifier`

`HistGradientBoostingClassifier` implementa el algoritmo basado en histogramas de **LightGBM**:
1. Utilizando el dataset `load_breast_cancer()` (split $75\% / 25\%$ estratificado), configura un `HistGradientBoostingClassifier` con:
   - `max_iter=500`, `learning_rate=0.05`, `l2_regularization=2.0`
   - `early_stopping=True`, `validation_fraction=0.15`, `n_iter_no_change=15`, `random_state=42`
2. Entrena el modelo e imprime cuántas iteraciones reales ejecutó antes de detenerse (`n_iter_`).
3. Muestra la matriz de confusión en el conjunto de prueba.


In [ ]:
# TU CODIGO AQUI
# 1. Cargar load_breast_cancer y dividir en train/test
# 2. Configurar y ajustar HistGradientBoostingClassifier con early_stopping=True
# 3. Imprimir modelo.n_iter_ y exactitud en test
# 4. Graficar ConfusionMatrixDisplay


---
### Ejercicio 4: Manejo de Desbalance de Clases con Pesos (`class_weight='balanced'`)

En detección de fraude o diagnóstico médico, la clase positiva suele ser minoritaria (en XGBoost se controla con `scale_pos_weight` y en LightGBM/HistGradientBoosting con `class_weight='balanced'`):
1. Genera un dataset sintético desbalanceado ($93\%$ clase 0, $7\%$ clase 1) usando:
   `X_imb, y_imb = make_classification(n_samples=2000, n_features=15, weights=[0.93, 0.07], random_state=42)`
2. Divide en train/test ($75\% / 25\%$, estratificado).
3. Entrena dos modelos `HistGradientBoostingClassifier(random_state=42)`: uno sin balanceo (`class_weight=None`) y otro con `class_weight='balanced'`.
4. Compara el **Recall** y el **F1-Score** de la clase minoritaria (clase 1) entre ambos modelos.


In [ ]:
# TU CODIGO AQUI
# 1. Generar dataset desbalanceado y dividir en train/test
# 2. Entrenar modelo base (class_weight=None) y modelo balanceado (class_weight='balanced')
# 3. Imprimir classification_report o F1/Recall de ambos modelos para la clase 1


---
### Ejercicio 5: Optimización de Hiperparámetros con `GridSearchCV`

Usando el conjunto de entrenamiento de `load_wine` del Ejercicio 2:
1. Crea una búsqueda en rejilla (`GridSearchCV` con `cv=5` y `scoring='accuracy'`) para `GradientBoostingClassifier(random_state=42)` explorando:
   - `'n_estimators': [50, 100]`
   - `'learning_rate': [0.05, 0.2]`
   - `'max_depth': [2, 4]`
   - `'subsample': [0.8, 1.0]`
2. Imprime la mejor combinación de hiperparámetros (`best_params_`) y la exactitud del mejor modelo en el conjunto de prueba.


In [ ]:
# TU CODIGO AQUI
# 1. Definir diccionario param_grid
# 2. Ejecutar GridSearchCV(cv=5) sobre X_train, y_train de Wine
# 3. Imprimir best_params_, best_score_ y exactitud final en X_test
